# T18 -- Lung-Region Attention Module (Candidate A)
## DenseNet121 + supervised spatial attention, trained on the COVID-19 Radiography Database

**Owner:** Member 1 | **Week 2, Milestone M2** | Novel contribution (Candidate A of 3)

**Design doc:** `Claude Working Files/T18_Lung_Region_Attention_WBS.md` -- read sections 1-7
before touching this notebook; this file follows that design exactly, section by section
(S0-S16). Where this notebook and `Member1_Guide.md` disagree, the WBS wins (see WBS section 4).

**Platform:** Kaggle, T4 (confirmed in S0 -- no sm_60 / P100 compatibility pin needed).

**S0 pre-flight -- resolved decisions (see WBS section 12.5 / chat log for full reasoning):**
- Lung masks: the COVID-19 Radiography Dataset's bundled `masks/` folder (confirmed official,
  not a fallback -- same folder structure locally and on the Kaggle-hosted dataset).
- Faithfulness (Grad-CAM EIL) definition: Member 5's T30 definition is not yet in the repo
  (checked `notebooks/candidate-c-grad-cam-shortcut-suppression-loss.ipynb` -- it has a
  training-time suppression loss on raw activations, not a post-hoc scoring function). Using
  our own documented definition (WBS section 6.3); swap later if T30's differs -- it's a
  metric function, not a training-time dependency.
- Hyperparameters: T13 (Member 2's DenseNet121 HP tuning) has no committed winner --
  `notebooks/baseline-cnn-model-dnn-research.ipynb` only exposes phase1_lr/phase2_lr as
  argparse *defaults* (1e-3/1e-5), it never sweeps them. Adopting T16's measured winner
  instead (WBS section 3.4a): `phase1_lr=3e-4, phase2_lr=3e-5, weight_decay=1e-3`.
- Mask-provenance wording (proposal says "automatically generated"; we use the dataset's
  supplied masks) recorded as a known discrepancy for the module card -- not blocking.


## Environment check

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    major, minor = torch.cuda.get_device_capability(0)
    print(f"Compute capability: sm_{major}{minor}")
    if (major, minor) == (6, 0):
        print("WARNING: Tesla P100 (sm_60) detected -- the AuxSeg notebook's torch==2.8.0+cu126 "
              "pin is required before any further torch import. This notebook was designed for "
              "T4 (S0) and does NOT include that pin by default.")


## S1 -- Config

Loads `configs/densenet121_lung_attention.yaml` (arm A2's config). Other arms are produced
from this base config via `merge_overrides()` at the point each arm is trained (S8/S9/S10) --
see the WBS's arm-to-flags table (Appendix A.3) for the exact overrides per arm.


In [ ]:
import sys
from pathlib import Path

# On Kaggle the repo isn't on sys.path by default. Upload the repo as a Kaggle Dataset
# (or clone it in a setup cell) and adjust this path, or run this notebook from a local
# clone where the repo root is already the working directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    # Kaggle fallback -- adjust to wherever the repo dataset/clone actually lands.
    REPO_ROOT = Path("/kaggle/working/Chest-X-ray-Disease-Detection")
sys.path.insert(0, str(REPO_ROOT))

from src.utils.config import load_config, merge_overrides

cfg = load_config(REPO_ROOT / "configs" / "densenet121_lung_attention.yaml")
print("Loaded config for experiment:", cfg["experiment"]["name"])
import json
print(json.dumps(cfg, indent=2))


## S2 -- Data layer -- mask-paired dataset + verification

Copy `JointTransform`, `CXRWithMaskDataset`, `stratified_split`, `build_dataloaders`, `compute_class_weights` from the AuxSeg notebook verbatim, then add reproducible worker seeding (WBS section 4.8). Run the Appendix A.1 verification + alignment-eyeball cells before proceeding -- a misaligned mask silently invalidates every downstream number.

This is also where `artifacts/splits/split_manifest_v1.csv` gets emitted (S1 step 5 of the WBS) using the real dataset.

## S3 -- The Lung-Region Attention module + CPU unit tests

`LungRegionAttention`, `attention_guidance_loss`, `compute_total_loss` (WBS Appendix A.2). Canonical copy lives in `src/modules/lung_attention.py`; `tests/test_lung_attention.py` must be green before any GPU work starts.

## S4 -- Assemble the model + parity check against vanilla

`DenseNetLungAttention` wrapper (WBS Appendix A.3), the backbone-agnostic freeze/unfreeze registry (Appendix A.4), and the CBAM comparator for arm A5 (Appendix A.4b). Parity check: gate_mode='none' must produce bit-identical logits to plain timm DenseNet121.

## S5 -- Loss + metrics implementation

`src/modules/attention_metrics.py` (ILAR, IoU, Dice, entropy, background attention) and the Grad-CAM EIL harness (pre-gate / post-gate taps, WBS section 6.3).

## S6 -- Training loop

Adapted `run_epoch` / `train_phase` / `evaluate` from AuxSeg: 3-tuple batches, AMP, per-epoch val precision/recall/F1/AUROC logged to W&B, `log_summary_metrics` at the end of each phase (WBS section S6 / policy Required Information).

## S7 -- Smoke test + overfit test

Do not proceed to S8 until: (1) the smoke run completes end to end, (2) the 32-image overfit test reaches 100% train accuracy with att_loss falling from ~0.6931 toward the ~0.2 entropy floor (WBS section 4.4a), (3) the lambda-sensitivity spot check shows higher ILAR at lambda=5 than lambda=0.

## S8 -- Arm A0 -- the vanilla control (full schedule)

`use_attention=False`. Sanity gate: judge against the measured 90.5-92% macro-F1 range (WBS section 2.4), not a single point estimate from the possibly-placeholder proposal table.

## S9 -- Lambda sweep (short schedule) + pre-registered selection

Mirrors T16's TUNING_CONFIGS -> tuning_summary.csv -> selected_config.json pattern exactly (WBS section S9). Test set stays out of scope for this entire section.

## S10 -- Full-schedule runs -- A1, A4, A2, A5 (and optionally A3)

Priority order: A2 (headline) -> A1 (isolates supervision) -> A5 (CBAM, required by the assignment brief section 9) -> A4 (isolates gating) -> A3 (optional). Re-seed at the start of every arm.

## S11 -- Evaluation, faithfulness, comparison table

Grad-CAM EIL at both taps, attention metrics, `T18_comparison_table.csv`, explicit pass/fail verdicts against acceptance criteria A1-A6, per-image prediction CSVs for Member 3's statistics (T37).

## S12 -- Figures

Heat-map overlays (3 per class), the attention grid, the lambda-sweep trade-off plot.

## S13 -- Efficiency measurement (A6)

`kusal-notebooks/efficiency.py::benchmark_model`, wrapped in `LogitsOnly` -- verify against the analytic estimate (+131,329 params, +0.22% GFLOPs).

## S14 -- Multi-seed repeat (optional, budget permitting)

A0 and A2 only, seeds 123 and 2026, full schedule. Explicitly state single-seed in the module card if this doesn't run.

## S15 -- Package and hand off

`T18_module_card.md`, deliverables to Member 3 / Member 2 / Member 5, the `LogitsOnly` wrapper note, RSNA class-mapping note, contribution list for the colour-highlighted paper.

## S16 -- Paper material

~400 words Methods, ~250 words Results -- draft in WBS Appendix B, edit against the actual `T18_comparison_table.csv` numbers before handing off.